# SI — example fits of Nile Red on α-synuclein aggregates (Ximea)

**Makes up:** Figure S87, "Fitting Nile Red localisations of α-Synuclein data" (`fig:Example_Asyn_Fit`, folder
`Example_Asyn_Puncta`): `example_puncta_raw_weights_fit_residual_spectrum_AsynPAINT`.

Six single-frame fits from the α-synuclein Nile Red PAINT acquisition of Figure 5 k-m: the raw punctum, the
final-pass fitting weights, the fitted model (x = fitted position), the residual, and the fitted spectral fingerprint
(normalised `A_B`, `A_G`, `A_R`, with their errors), in the published layout (23.8 mm square panels).

**Bayer key:** each raw punctum carries the 2 x 2 key of the colour filters under their top-left pixels.
The masks are those of the camera chip at the acquisition's real position (the camera ROI read from the
MicroManager metadata), as are the calibration maps, exactly as in the fit.

**Data:** BioStudies `S-BIAD3210`, `2026_Multicolour_Paper/Ximea/AlphaSynPAINT/20260205_alpha_syn_NR_PLL/1nM_NR_10uM_asyn72h_2p5AuNP/50ms_70p_515_LP515_bp650_200_4/`
(8 TIFFs, ~34 GB, and the MicroManager metadata), fitted with the settings of Figure 5 k-m (pfa 1e-3, peak
wavelength 0.65 um, `ROI_size` 12). Each example is shown as the 12 x 12 px crop around its fit.

**Which fits:** drawn at random (`SELECTION_SEED`) from those that pass the Methods quality cuts (> 50 detected
photoelectrons, chi^2 < 3, PSF sigma 75-250 nm as in Figure 5 k-m, position error < 1 px), have colour errors
< 0.05 (as the other example-fit figures) and 200-590 detected photoelectrons (the middle half of this acquisition's
single-frame fits that pass the Methods cuts). Nile Red's colour depends on its environment, so there is no `A_R`
window.

**Units:** everything shown is in photoelectrons (pe); see `_example_puncta.py` for how the fit database's
columns convert.

## FAST_MODE

`FAST_MODE = True` (default) downloads and fits only the first TIFF (~4 GB), and draws the examples from it.
`FAST_MODE = False` downloads and fits all 8 (~34 GB), as for Figure 5 k-m.


---
## 0 — Imports

In [ ]:
import json
import sys
import types
from pathlib import Path

import numpy as np
import pandas as pd

# _biostudies_download.py and _example_puncta.py live in the parent folder (notebooks/figures/).
sys.path.insert(0, str(Path.cwd().resolve().parent))
import _biostudies_download as bd
import _example_puncta as ep

from pyS3M import IOFunctions
from pyS3M import sCMOSFunctions
from pyS3M import SR_Functions
from pyS3M import PlottingBase

IO = IOFunctions.IO_Functions()
sCMOS = sCMOSFunctions.sCMOS_Functions()
SupRes_F = SR_Functions.SuperRes_Functions()   # Ximea defaults (69 nm pixels)
plotter = PlottingBase.PublicationPlotter(dark_background=False)

print("Python:", sys.version)

---
## 1 — Configuration

In [ ]:
FAST_MODE = True   # True: first TIFF only (~4 GB). False: all 8 TIFFs (~34 GB), as for Figure 5 k-m.
REFIT = False      # True: fit again even if a fit with the same files and settings exists.

PIXEL_SIZE_NM = 69.0   # Ximea camera pixel

REPO_ROOT = Path.cwd().resolve().parents[2]   # notebooks/figures/SI -> repo root
CALIB_DIR = REPO_ROOT / "Camera_Calibrations" / "Ximea_Camera"
OUT_ROOT  = Path("outputs") / "FigureSI_ExampleFitting_AlphaSyn"   # git-ignored (bare outputs/ rule)
FIG_DIR   = OUT_ROOT / "figures"
# Point DATA_DIR at a drive with room for the download. The fit reads every TIFF in the folder,
# so keep one folder per FAST_MODE setting.
DATA_DIR  = OUT_ROOT / ("data_fast" if FAST_MODE else "data_full")
FIG_DIR.mkdir(parents=True, exist_ok=True)
DATA_DIR.mkdir(parents=True, exist_ok=True)

calib = dict(
    gain_map=IO.read_tiff(CALIB_DIR / "gain.tif"),
    offset_map=IO.read_tiff(CALIB_DIR / "offset.tif"),
    rqe=IO.read_tiff(CALIB_DIR / "rqe.tif"),
    read_noise=IO.read_tiff(CALIB_DIR / "readnoise.tif"),
    variance=IO.read_tiff(CALIB_DIR / "variance.tif"),
)

smoothing_function = types.SimpleNamespace()
smoothing_function.args = {"sigma": 1.5}
smoothing_function.extent = 1.5
smoothing_function.smoothing_function = sCMOS.gaussian_filter_stack
smoothing_function.data_arg = "image"

# The fit settings of Figure 5 k-m
FIT_PARAMS = dict(pfa=1e-3, peak_wavelength=0.65, ROI_size=12, NA=1.49)
DISPLAY_SIZE = 12   # px shown around each fit

N_PUNCTA = 6         # example fits, one per column
SELECTION_SEED = 8   # seed of the example draw
# Methods quality cuts (PSF sigma 75-250 nm, as Figure 5 k-m), 200-590 detected photoelectrons (middle half of the
# single-frame fits that pass the Methods cuts), and colour errors < 0.05
QC_QUERY = ("photoelectrons > 200 and photoelectrons < 590 and chi_sqr < 3 and xc_err < 1 and yc_err < 1"
            " and A_B_err < 0.05 and A_G_err < 0.05 and A_R_err < 0.05"
            f" and {75 / PIXEL_SIZE_NM} < s_x < {250 / PIXEL_SIZE_NM}"
            f" and {75 / PIXEL_SIZE_NM} < s_y < {250 / PIXEL_SIZE_NM}")

print(f"FAST_MODE  : {FAST_MODE}")
print(f"Data dir   : {DATA_DIR.resolve()}")
print(f"Figures dir: {FIG_DIR.resolve()}")

---
## 2 — Download and fit

Downloads the acquisition (or its first TIFF) from BioStudies with its metadata, and fits it with
`fit_imaging_data` (one `Localisations.h5`, frames numbered across the TIFFs). Re-running is safe: complete
downloads are skipped, and the fit is reused unless the files or settings change (or `REFIT = True`).

In [ ]:
REMOTE_FOLDER = "2026_Multicolour_Paper/Ximea/AlphaSynPAINT/20260205_alpha_syn_NR_PLL/1nM_NR_10uM_asyn72h_2p5AuNP/50ms_70p_515_LP515_bp650_200_4"
downloaded = bd.download_acquisition(REMOTE_FOLDER, DATA_DIR, first_file_only=FAST_MODE)

fit_file = DATA_DIR / "Localisations.h5"
fit_record_file = DATA_DIR / "Localisations_fit_record.json"
fit_record = {"images": [p.name for p in downloaded["images"]], **FIT_PARAMS}
previous = json.loads(fit_record_file.read_text()) if fit_record_file.exists() else None
if REFIT or not fit_file.exists() or previous != fit_record:
    SupRes_F.fit_imaging_data(
        DATA_DIR, smoothing_function, calib["gain_map"], calib["offset_map"], calib["rqe"], calib["read_noise"],
        variance=calib["variance"], image_type=".tif", **FIT_PARAMS,
    )
    fit_record_file.write_text(json.dumps(fit_record, indent=1))
else:
    print(f"Reusing {fit_file} (same files and settings)")

In [ ]:
# The camera ROI of the acquisition: calibration maps and Bayer masks are cropped to it, as in the fit,
# so each punctum's Bayer key shows the filters actually under it.
start_x, start_y, width, height = IO.metadata_reader_imageJ(downloaded["metadata"][0])
cropped = SupRes_F.helper.crop_calibration_maps(calib, start_x, start_y, width, height)
masks = SupRes_F.mask.get_stacked_masks(start_x, start_y, width, height, SupRes_F.mosaic_unit)

# The TIFFs in the order the fit numbered their frames
reader = ep.FrameReader(SupRes_F.helper.file_search(DATA_DIR, ".tif", ""))

columns = ["xc", "yc", "s_x", "s_y", "frame", "chi_sqr", "xc_err", "yc_err", "A_B", "A_G", "A_R",
           "A_B_err", "A_G_err", "A_R_err", "bg_B", "bg_G", "bg_R",
           "photoelectrons", "background_photoelectrons"]
fits_df = pd.read_hdf(fit_file, key="data", columns=columns)
print(f"ROI start ({start_x}, {start_y}), {width} x {height} px; {len(reader.files)} TIFF(s)")
print(f"{len(fits_df):,} single-frame fits over {fits_df['frame'].nunique():,} frames")

---
## 3 — Example fits

`N_PUNCTA` fits (with a full fitting ROI inside the frame) drawn at random (`SELECTION_SEED`) from those that pass
the quality cuts. For each: the raw punctum in photoelectrons with its Bayer key, the weights of the fit's final
pass (1 / (max(pe, 0) + 1 + read noise^2), pe^-2), the fitted model rebuilt from the stored parameters, the
residual (raw - fit), and the fitted spectral fingerprint, on the `DISPLAY_SIZE` crop. Drawn in the published
layout (`ep.plot_example_grid`).

In [ ]:
chosen = ep.choose_puncta(fits_df, width, height, FIT_PARAMS["ROI_size"], N_PUNCTA, SELECTION_SEED, query=QC_QUERY)
puncta = ep.build_example_puncta(chosen, reader, cropped, masks, DISPLAY_SIZE)
print(chosen[["frame", "xc", "yc", "photoelectrons", "background_photoelectrons",
              "A_B", "A_G", "A_R", "chi_sqr"]].round(3).to_string())

fig, axs = ep.plot_example_grid(plotter, puncta, PIXEL_SIZE_NM)
plotter.save_or_show(fig, save_path=str(FIG_DIR / "example_puncta_raw_weights_fit_residual_spectrum_AsynPAINT.svg"),
                     show=True)